In [1]:
import geopandas as gpd
import joblib
import numpy as np
import odc.geo  # noqa: F401
import pandas as pd
from shapely import geometry
from sklearn.ensemble import RandomForestClassifier

from utils import load_data

In [2]:
%reload_ext autoreload
%autoreload 2

## Find and load S2 data

Load data and set up your array to use for prediction

In [3]:
# Configure some things up front
chunks = dict(x=256, y=256)
datetime = "2022"

bbox_fiji = [177.14, -18.41, 178.80, -17.2]
bbox = [178.14, -17.81, 178.50, -17.4]
bbox_geometry = geometry.box(*bbox)

gdf = gpd.GeoDataFrame({'geometry': [bbox_geometry]}, crs='EPSG:4326')
gdf.explore()

In [4]:
merged = load_data(bbox, chunks=dict(x=256, y=256), resolution=10)

Successfully found and loading 4 tiles for the year 2022...


/srv/conda/envs/notebook/lib/python3.11/site-packages/rasterio/warp.py:387: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


Searching for Sentinel-1 radar data...
Total S1 items returned for bbox: 32
Sample Item ID: dep_s1_geomad_064_021_2024
Sample Item Properties: {'created': '2025-11-06T23:58:05.769540Z', 'datetime': '2024-01-01T00:00:00Z', 'proj:bbox': [3144000.0, -1984000.0, 3240000.0, -1888000.0], 'proj:shape': [9600, 9600], 'end_datetime': '2024-12-31T23:59:59Z', 'proj:geometry': {'type': 'Polygon', 'coordinates': [[[3144000.0, -1984000.0], [3240000.0, -1984000.0], [3240000.0, -1888000.0], [3144000.0, -1888000.0], [3144000.0, -1984000.0]]]}, 'proj:transform': [10.0, 0.0, 3144000.0, 0.0, -10.0, -1888000.0, 0.0, 0.0, 1.0], 'start_datetime': '2024-01-01T00:00:00.000Z', 'proj:code': 'EPSG:3832'}
-> Successfully found and filtered 4 radar tiles for 2022...
Successfully merged Sentinel-1 data into master dataset.


In [5]:
merged

<xarray.Dataset> Size: 2GB
Dimensions:        (y: 4760, x: 4008)
Coordinates:
  * y              (y) float64 38kB -1.955e+06 -1.955e+06 ... -2.002e+06
  * x              (x) float64 32kB 3.133e+06 3.133e+06 ... 3.173e+06 3.173e+06
    spatial_ref    int32 4B 3832
    time           datetime64[ns] 8B 2022-01-01
Data variables: (12/28)
    blue           (y, x) uint16 38MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    green          (y, x) uint16 38MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    red            (y, x) uint16 38MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    rededge1       (y, x) uint16 38MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    rededge2       (y, x) uint16 38MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    rededge3       (y, x) uint16 38MB dask.array<chunksize=(256, 256), meta=np.ndarray>
    ...             ...
    elevation_std  (y, x) float32 76MB dask.array<chunksize=(257, 257), meta=np.ndarray>
    slope          (y, x) float32 76MB nan nan nan 7.099 ... nan nan nan nan
    aspect         (y, x) float32 76MB nan nan nan 272.4 ... nan nan nan nan
    mean_vv        (y, x) float32 76MB 0.2201 0.2241 0.2251 ... 0.1464 0.09897
    mean_vh        (y, x) float32 76MB 0.04241 0.04395 ... 0.0315 0.02406
    mean_vv_vh     (y, x) float32 76MB 5.191 5.1 5.248 ... 4.621 4.647 4.113

## Train and predict

When you change your training data, you can re-train and predict here.

In [6]:
training_file = "training_data/mred-tdata.geojson"

tdata = gpd.read_file(training_file, bbox=bbox_geometry)
tdata.explore()

In [7]:
# Get values for each of the image bands at each of the points.
pts_proj = tdata.to_crs(merged.odc.crs)

# a DataArray with x & y coords
pts_da = pts_proj.assign(x=pts_proj.geometry.x, y=pts_proj.geometry.y).to_xarray()

# a dataframe or series (for a single point)
pt_values_i = (
    merged.sel(pts_da[["x", "y"]], method="nearest").squeeze().compute().to_pandas()
)

if isinstance(pt_values_i, pd.Series):
    pt_values_i = pt_values_i.to_frame().transpose()
    pt_values_i.index = tdata.index

In [8]:
training_array = pd.concat([tdata, pt_values_i], axis=1).to_crs(4326)
training_array = training_array.drop(
    columns=[
        "y",
        "x",
        "spatial_ref",
        "time",
        "fid",
        "index",
        "lulc_class",
        "path",
        "geometry",
        "layer",
        "id",
    ]
)
# Drop rows where there are any NaNs
training_array = training_array.dropna()

training_array.head()

,lulc_code,blue,green,red,rededge1,rededge2,rededge3,nir,swir16,swir22,...,ndbi,ndwi,ndre,elevation,elevation_std,slope,aspect,mean_vv,mean_vh,mean_vv_vh
50,7.0,647.0,923.0,664.0,1286.0,2815.0,3302.0,3492.0,1822.0,1058.0,...,0.372836,14.262061,0.461699,15.229146,1.335231,5.488382,231.532440,0.229282,0.044159,5.192220
53,7.0,423.0,725.0,358.0,1134.0,3868.0,4925.0,5089.0,2155.0,986.0,...,0.546581,10.521500,0.635546,215.913071,0.812147,11.223964,292.107697,0.439720,0.083388,5.273170
54,7.0,790.0,886.0,880.0,830.0,1066.0,1335.0,935.0,930.0,572.0,...,0.124473,35.962109,0.059490,12.929992,3.738234,12.274364,144.292786,0.049139,0.016438,2.989280
55,2.0,600.0,883.0,704.0,1398.0,2598.0,2943.0,2776.0,2072.0,1242.0,...,0.300300,17.393550,0.330139,5.190803,0.048622,0.420869,257.989716,0.232699,0.053312,4.364868
58,2.0,557.0,914.0,537.0,1486.0,3685.0,4295.0,4000.0,2075.0,1062.0,...,0.425256,12.708588,0.458257,6.554387,0.026741,0.721626,153.558151,0.242831,0.057771,4.203316


In [9]:
# 1. Check feature names
print("Model expected features:", getattr(model, "feature_names_in_", "None saved"))

# 2. Check for constant or null features in training data
for i, col in enumerate(training_array.columns if hasattr(training_array, "columns") else range(training_array.shape[1])):
    vals = training_array[:, i] if not hasattr(training_array, "columns") else training_array[col]
    print(f"Feature {col}: min={vals.min()}, max={vals.max()}, nans={np.isnan(vals).sum()}")

NameError: name 'model' is not defined

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    n_jobs=-1,
    random_state=42,
)

training_data = np.array(training_array)[:, 1:]
classes = np.array(training_array)[:, 0]

model = classifier.fit(training_data, classes)


In [ ]:
joblib.dump(model, "test_model.dump")

In [65]:
# Print feature importances against column headings
fields_importances = sorted(
    zip(training_array.columns[1:], classifier.feature_importances_),
    key=lambda x: x[1],
    reverse=True,
)

for i in fields_importances:
    # Format as a table to 2 decinal places
    print(f"{i[0]:<11}| {i[1]:.2f}")


blue       | 0.00
green      | 0.00
red        | 0.00
rededge1   | 0.00
rededge2   | 0.00
rededge3   | 0.00
nir        | 0.00
swir16     | 0.00
swir22     | 0.00
emad       | 0.00
bcmad      | 0.00
smad       | 0.00
ndvi       | 0.00
mndwi      | 0.00
evi        | 0.00
savi       | 0.00
bsi        | 0.00
ndmi       | 0.00
ndbi       | 0.00
ndwi       | 0.00
ndre       | 0.00
elevation  | 0.00
elevation_std| 0.00
slope      | 0.00
aspect     | 0.00
mean_vv    | 0.00
mean_vh    | 0.00
mean_vv_vh | 0.00
